<a href="https://colab.research.google.com/github/jennyk23/Magpy/blob/main/atividade5_clinica_medica_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade 5 — Sistema de Clínica Médica
Execute a única célula abaixo. Ela instala o MariaDB, cria o banco, insere os dados e exibe os resultados.

In [1]:
# ATIVIDADE 5 – EXECUÇÃO COMPLETA NO GOOGLE COLAB

import os
import shutil
import subprocess
import time
from pathlib import Path

if shutil.which("mariadb") is None:
    print("Instalando MariaDB...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "mariadb-server", "mariadb-client"],
        check=True,
        stdout=subprocess.DEVNULL
    )

os.makedirs("/run/mysqld", exist_ok=True)
subprocess.run(["chown", "mysql:mysql", "/run/mysqld"], check=False)
subprocess.run(
    ["service", "mariadb", "start"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(20):
    teste = subprocess.run(
        ["mariadb-admin", "-u", "root", "ping", "--silent"],
        text=True,
        capture_output=True
    )
    if teste.returncode == 0:
        break
    time.sleep(1)
else:
    raise RuntimeError("Não foi possível iniciar o MariaDB.")

print("MariaDB iniciado com sucesso.")

sql = "-- ============================================================\n-- ATIVIDADE 5 – SISTEMA DE CLÍNICA MÉDICA\n-- MYSQL / MARIADB\n-- ============================================================\n\nDROP DATABASE IF EXISTS clinica_medica;\nCREATE DATABASE clinica_medica CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;\nUSE clinica_medica;\n\nCREATE TABLE paciente (\n    id_paciente INT AUTO_INCREMENT,\n    nome VARCHAR(150) NOT NULL,\n    cpf VARCHAR(14) NOT NULL,\n    data_nascimento DATE,\n    telefone VARCHAR(20),\n    email VARCHAR(150),\n    CONSTRAINT pk_paciente PRIMARY KEY (id_paciente),\n    CONSTRAINT uq_paciente_cpf UNIQUE (cpf),\n    CONSTRAINT uq_paciente_email UNIQUE (email),\n    CONSTRAINT chk_paciente_nome CHECK (CHAR_LENGTH(TRIM(nome)) > 0)\n) ENGINE=InnoDB;\n\nCREATE TABLE medico (\n    id_medico INT AUTO_INCREMENT,\n    nome VARCHAR(150) NOT NULL,\n    crm VARCHAR(30) NOT NULL,\n    telefone VARCHAR(20),\n    email VARCHAR(150),\n    CONSTRAINT pk_medico PRIMARY KEY (id_medico),\n    CONSTRAINT uq_medico_crm UNIQUE (crm),\n    CONSTRAINT uq_medico_email UNIQUE (email),\n    CONSTRAINT chk_medico_nome CHECK (CHAR_LENGTH(TRIM(nome)) > 0)\n) ENGINE=InnoDB;\n\nCREATE TABLE especialidade (\n    id_especialidade INT AUTO_INCREMENT,\n    nome VARCHAR(100) NOT NULL,\n    descricao VARCHAR(255),\n    CONSTRAINT pk_especialidade PRIMARY KEY (id_especialidade),\n    CONSTRAINT uq_especialidade_nome UNIQUE (nome)\n) ENGINE=InnoDB;\n\nCREATE TABLE medico_especialidade (\n    id_medico INT NOT NULL,\n    id_especialidade INT NOT NULL,\n    CONSTRAINT pk_medico_especialidade PRIMARY KEY (id_medico, id_especialidade),\n    CONSTRAINT fk_me_medico FOREIGN KEY (id_medico)\n        REFERENCES medico(id_medico) ON UPDATE CASCADE ON DELETE CASCADE,\n    CONSTRAINT fk_me_especialidade FOREIGN KEY (id_especialidade)\n        REFERENCES especialidade(id_especialidade) ON UPDATE CASCADE ON DELETE CASCADE\n) ENGINE=InnoDB;\n\nCREATE TABLE consulta (\n    id_consulta INT AUTO_INCREMENT,\n    id_paciente INT NOT NULL,\n    id_medico INT NOT NULL,\n    data_hora DATETIME NOT NULL,\n    motivo VARCHAR(255) NOT NULL,\n    diagnostico VARCHAR(500),\n    status VARCHAR(20) NOT NULL DEFAULT 'Agendada',\n    CONSTRAINT pk_consulta PRIMARY KEY (id_consulta),\n    CONSTRAINT chk_consulta_status CHECK (status IN ('Agendada','Realizada','Cancelada')),\n    CONSTRAINT fk_consulta_paciente FOREIGN KEY (id_paciente)\n        REFERENCES paciente(id_paciente) ON UPDATE CASCADE ON DELETE RESTRICT,\n    CONSTRAINT fk_consulta_medico FOREIGN KEY (id_medico)\n        REFERENCES medico(id_medico) ON UPDATE CASCADE ON DELETE RESTRICT\n) ENGINE=InnoDB;\n\nCREATE TABLE exame (\n    id_exame INT AUTO_INCREMENT,\n    id_consulta INT NOT NULL,\n    nome_exame VARCHAR(150) NOT NULL,\n    data_solicitacao DATE NOT NULL,\n    data_realizacao DATE,\n    status VARCHAR(20) NOT NULL DEFAULT 'Solicitado',\n    CONSTRAINT pk_exame PRIMARY KEY (id_exame),\n    CONSTRAINT chk_exame_status CHECK (status IN ('Solicitado','Realizado','Cancelado')),\n    CONSTRAINT fk_exame_consulta FOREIGN KEY (id_consulta)\n        REFERENCES consulta(id_consulta) ON UPDATE CASCADE ON DELETE CASCADE\n) ENGINE=InnoDB;\n\nCREATE TABLE resultado (\n    id_resultado INT AUTO_INCREMENT,\n    id_exame INT NOT NULL,\n    descricao TEXT NOT NULL,\n    data_resultado DATE NOT NULL,\n    observacao VARCHAR(500),\n    CONSTRAINT pk_resultado PRIMARY KEY (id_resultado),\n    CONSTRAINT uq_resultado_exame UNIQUE (id_exame),\n    CONSTRAINT fk_resultado_exame FOREIGN KEY (id_exame)\n        REFERENCES exame(id_exame) ON UPDATE CASCADE ON DELETE CASCADE\n) ENGINE=InnoDB;\n\nCREATE TABLE receita (\n    id_receita INT AUTO_INCREMENT,\n    id_consulta INT NOT NULL,\n    data_emissao DATE NOT NULL,\n    observacao VARCHAR(500),\n    CONSTRAINT pk_receita PRIMARY KEY (id_receita),\n    CONSTRAINT fk_receita_consulta FOREIGN KEY (id_consulta)\n        REFERENCES consulta(id_consulta) ON UPDATE CASCADE ON DELETE CASCADE\n) ENGINE=InnoDB;\n\nCREATE TABLE medicamento (\n    id_medicamento INT AUTO_INCREMENT,\n    nome VARCHAR(150) NOT NULL,\n    principio_ativo VARCHAR(150) NOT NULL,\n    concentracao VARCHAR(80),\n    forma_farmaceutica VARCHAR(50),\n    CONSTRAINT pk_medicamento PRIMARY KEY (id_medicamento),\n    CONSTRAINT uq_medicamento UNIQUE (nome, concentracao, forma_farmaceutica)\n) ENGINE=InnoDB;\n\nCREATE TABLE receita_medicamento (\n    id_receita INT NOT NULL,\n    id_medicamento INT NOT NULL,\n    dosagem VARCHAR(100) NOT NULL,\n    frequencia VARCHAR(100) NOT NULL,\n    duracao VARCHAR(100) NOT NULL,\n    quantidade INT NOT NULL,\n    orientacao VARCHAR(255),\n    CONSTRAINT pk_receita_medicamento PRIMARY KEY (id_receita, id_medicamento),\n    CONSTRAINT chk_rm_quantidade CHECK (quantidade > 0),\n    CONSTRAINT fk_rm_receita FOREIGN KEY (id_receita)\n        REFERENCES receita(id_receita) ON UPDATE CASCADE ON DELETE CASCADE,\n    CONSTRAINT fk_rm_medicamento FOREIGN KEY (id_medicamento)\n        REFERENCES medicamento(id_medicamento) ON UPDATE CASCADE ON DELETE RESTRICT\n) ENGINE=InnoDB;\n\nINSERT INTO paciente (nome, cpf, data_nascimento, telefone, email) VALUES\n('Ana Beatriz Souza','111.111.111-11','1995-05-12','(65) 99999-1111','ana.souza@email.com'),\n('Carlos Henrique Lima','222.222.222-22','1988-09-03','(65) 99999-2222','carlos.lima@email.com'),\n('Mariana Alves Costa','333.333.333-33','2001-01-20','(65) 99999-3333','mariana.costa@email.com');\n\nINSERT INTO medico (nome, crm, telefone, email) VALUES\n('Dra. Juliana Ferreira','CRM-MT 12345','(65) 98888-1111','juliana.ferreira@clinica.com'),\n('Dr. Roberto Martins','CRM-MT 23456','(65) 98888-2222','roberto.martins@clinica.com'),\n('Dra. Fernanda Oliveira','CRM-MT 34567','(65) 98888-3333','fernanda.oliveira@clinica.com');\n\nINSERT INTO especialidade (nome, descricao) VALUES\n('Clínica Geral','Atendimento médico geral'),\n('Cardiologia','Diagnóstico e tratamento de doenças do coração'),\n('Neurologia','Diagnóstico e tratamento do sistema nervoso'),\n('Pediatria','Atendimento médico de crianças e adolescentes');\n\nINSERT INTO medico_especialidade VALUES\n(1,1),(1,2),(2,3),(3,1),(3,4);\n\nINSERT INTO consulta (id_paciente,id_medico,data_hora,motivo,diagnostico,status) VALUES\n(1,1,'2026-06-20 08:30:00','Dor no peito e cansaço','Necessidade de avaliação cardiovascular','Realizada'),\n(2,2,'2026-06-21 10:00:00','Dor de cabeça frequente','Cefaleia em investigação','Realizada'),\n(3,3,'2026-07-02 14:00:00','Consulta de rotina',NULL,'Agendada');\n\nINSERT INTO exame (id_consulta,nome_exame,data_solicitacao,data_realizacao,status) VALUES\n(1,'Eletrocardiograma','2026-06-20','2026-06-20','Realizado'),\n(1,'Hemograma completo','2026-06-20','2026-06-21','Realizado'),\n(2,'Ressonância magnética do crânio','2026-06-21','2026-06-25','Realizado'),\n(2,'Exame de sangue','2026-06-21','2026-06-22','Realizado');\n\nINSERT INTO resultado (id_exame,descricao,data_resultado,observacao) VALUES\n(1,'Ritmo sinusal regular, sem alterações agudas.','2026-06-20','Resultado dentro dos parâmetros esperados.'),\n(2,'Hemograma sem alterações significativas.','2026-06-21','Sem sinais de anemia ou infecção.'),\n(3,'Ausência de alterações estruturais relevantes.','2026-06-26','Acompanhamento clínico recomendado.'),\n(4,'Resultados laboratoriais dentro dos valores de referência.','2026-06-22','Sem observações adicionais.');\n\nINSERT INTO receita (id_consulta,data_emissao,observacao) VALUES\n(1,'2026-06-20','Uso conforme orientação médica.'),\n(2,'2026-06-21','Retornar em 15 dias para reavaliação.'),\n(2,'2026-06-21','Medicamento para uso em caso de dor.');\n\nINSERT INTO medicamento (nome,principio_ativo,concentracao,forma_farmaceutica) VALUES\n('AAS','Ácido acetilsalicílico','100 mg','Comprimido'),\n('Losartana','Losartana potássica','50 mg','Comprimido'),\n('Paracetamol','Paracetamol','750 mg','Comprimido'),\n('Dipirona','Dipirona monoidratada','500 mg','Comprimido'),\n('Omeprazol','Omeprazol','20 mg','Cápsula');\n\nINSERT INTO receita_medicamento\n(id_receita,id_medicamento,dosagem,frequencia,duracao,quantidade,orientacao) VALUES\n(1,1,'1 comprimido','Uma vez ao dia','30 dias',30,'Tomar após o almoço.'),\n(1,2,'1 comprimido','A cada 12 horas','30 dias',60,'Verificar a pressão regularmente.'),\n(2,5,'1 cápsula','Uma vez ao dia','14 dias',14,'Tomar em jejum.'),\n(3,3,'1 comprimido','A cada 8 horas se houver dor','5 dias',15,'Não ultrapassar a dose prescrita.'),\n(3,4,'1 comprimido','A cada 6 horas se necessário','3 dias',12,'Suspender em caso de reação alérgica.');\n"
arquivo_sql = Path("/content/atividade5_clinica_medica.sql")
arquivo_sql.write_text(sql, encoding="utf-8")
print("Arquivo SQL criado em:", arquivo_sql)

with arquivo_sql.open("r", encoding="utf-8") as arquivo:
    execucao = subprocess.run(
        ["mariadb", "-u", "root", "--default-character-set=utf8mb4"],
        stdin=arquivo,
        text=True,
        capture_output=True
    )

if execucao.returncode != 0:
    print("ERRO DO MYSQL/MARIADB:")
    print(execucao.stderr)
    raise RuntimeError("Erro ao executar a Atividade 5.")

print("Banco clinica_medica criado com sucesso.")

consultas = "USE clinica_medica;\n\nSELECT 'MÉDICOS E ESPECIALIDADES' AS etapa;\nSELECT m.id_medico, m.nome AS medico, m.crm,\n       GROUP_CONCAT(e.nome ORDER BY e.nome SEPARATOR ', ') AS especialidades\nFROM medico m\nJOIN medico_especialidade me ON me.id_medico = m.id_medico\nJOIN especialidade e ON e.id_especialidade = me.id_especialidade\nGROUP BY m.id_medico, m.nome, m.crm\nORDER BY m.id_medico;\n\nSELECT 'CONSULTAS' AS etapa;\nSELECT c.id_consulta, p.nome AS paciente, m.nome AS medico,\n       c.data_hora, c.motivo, c.diagnostico, c.status\nFROM consulta c\nJOIN paciente p ON p.id_paciente = c.id_paciente\nJOIN medico m ON m.id_medico = c.id_medico\nORDER BY c.id_consulta;\n\nSELECT 'EXAMES E RESULTADOS' AS etapa;\nSELECT e.id_exame, p.nome AS paciente, e.nome_exame,\n       e.data_solicitacao, e.data_realizacao, e.status,\n       r.descricao AS resultado, r.data_resultado\nFROM exame e\nJOIN consulta c ON c.id_consulta = e.id_consulta\nJOIN paciente p ON p.id_paciente = c.id_paciente\nLEFT JOIN resultado r ON r.id_exame = e.id_exame\nORDER BY e.id_exame;\n\nSELECT 'RECEITAS E MEDICAMENTOS' AS etapa;\nSELECT r.id_receita, p.nome AS paciente, m.nome AS medico,\n       med.nome AS medicamento, med.concentracao,\n       rm.dosagem, rm.frequencia, rm.duracao,\n       rm.quantidade, rm.orientacao\nFROM receita r\nJOIN consulta c ON c.id_consulta = r.id_consulta\nJOIN paciente p ON p.id_paciente = c.id_paciente\nJOIN medico m ON m.id_medico = c.id_medico\nJOIN receita_medicamento rm ON rm.id_receita = r.id_receita\nJOIN medicamento med ON med.id_medicamento = rm.id_medicamento\nORDER BY r.id_receita, med.nome;\n\nSELECT 'CONTAGEM FINAL' AS etapa;\nSELECT 'Pacientes' AS tabela, COUNT(*) AS total FROM paciente\nUNION ALL SELECT 'Médicos', COUNT(*) FROM medico\nUNION ALL SELECT 'Especialidades', COUNT(*) FROM especialidade\nUNION ALL SELECT 'Médico-Especialidade', COUNT(*) FROM medico_especialidade\nUNION ALL SELECT 'Consultas', COUNT(*) FROM consulta\nUNION ALL SELECT 'Exames', COUNT(*) FROM exame\nUNION ALL SELECT 'Resultados', COUNT(*) FROM resultado\nUNION ALL SELECT 'Receitas', COUNT(*) FROM receita\nUNION ALL SELECT 'Medicamentos', COUNT(*) FROM medicamento\nUNION ALL SELECT 'Receita-Medicamento', COUNT(*) FROM receita_medicamento;\n\nSELECT 'TABELAS CRIADAS' AS etapa;\nSHOW TABLES;\n"
resultado = subprocess.run(
    [
        "mariadb", "-u", "root", "--table",
        "--default-character-set=utf8mb4", "-e", consultas
    ],
    text=True,
    capture_output=True
)

print(resultado.stdout)

if resultado.returncode != 0:
    print("ERRO DO MYSQL/MARIADB:")
    print(resultado.stderr)
    raise RuntimeError("Erro ao consultar os resultados.")

print("Atividade 5 executada com sucesso.")


Instalando MariaDB...
MariaDB iniciado com sucesso.
Arquivo SQL criado em: /content/atividade5_clinica_medica.sql
Banco clinica_medica criado com sucesso.
+---------------------------+
| etapa                     |
+---------------------------+
| MÉDICOS E ESPECIALIDADES  |
+---------------------------+
+-----------+------------------------+--------------+-----------------------------+
| id_medico | medico                 | crm          | especialidades              |
+-----------+------------------------+--------------+-----------------------------+
|         1 | Dra. Juliana Ferreira  | CRM-MT 12345 | Cardiologia, Clínica Geral  |
|         2 | Dr. Roberto Martins    | CRM-MT 23456 | Neurologia                  |
|         3 | Dra. Fernanda Oliveira | CRM-MT 34567 | Clínica Geral, Pediatria    |
+-----------+------------------------+--------------+-----------------------------+
+-----------+
| etapa     |
+-----------+
| CONSULTAS |
+-----------+
+-------------+----------------------